In [1]:
!pip install langid -q

In [2]:
import pandas as pd
import numpy as np
import re
import os
import subprocess
import sys

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier, SGDClassifier

In [3]:
DATA_PATH = "/kaggle/input/datasets/arailymakhmet/merged/merged_dataset.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.head())
print(df["label"].value_counts())
print(df["lang_label"].value_counts())

df = df.dropna(subset=["text", "label"]).copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

(81648, 3)
                                                text  label lang_label
0  году специальную экономическую зону морпорт ак...      1         ru
1                      мынау жақсы екен орнатыңыздар      1         kk
2  посмотрела сорок восемь серий двадцать восьмой...      1         ru
3                                     гадость гадкая      0         ru
4  обычно всё устраивает это пельмени ломаные вар...      0         ru
label
1    40824
0    40824
Name: count, dtype: int64
lang_label
ru    40824
kk    40824
Name: count, dtype: int64


In [4]:

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (65295,)
Test: (16324,)


In [5]:
results = []

def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1_macro = f1_score(y_test, preds, average="macro")
    f1_weighted = f1_score(y_test, preds, average="weighted")

    results.append({
        "model": name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

    print("=" * 70)
    print(name)
    print("=" * 70)
    print("Accuracy:", acc)
    print("F1 macro:", f1_macro)
    print("F1 weighted:", f1_weighted)
    print("\nClassification report:")
    print(classification_report(y_test, preds))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, preds))

In [6]:
#TF-IDF + RidgeClassifier
ridge_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=150000,
        sublinear_tf=True
    )),
    ("clf", RidgeClassifier(
        alpha=1.0,
        class_weight="balanced"
    ))
])

ridge_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + RidgeClassifier",
    ridge_model,
    X_test,
    y_test
)

Word TF-IDF + RidgeClassifier
Accuracy: 0.8431144327370742
F1 macro: 0.843111295233526
F1 weighted: 0.8431113811925273

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      8163
           1       0.85      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6918 1245]
 [1316 6845]]


In [7]:
#TF-IDF + SGDClassifier
sgd_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=150000,
        sublinear_tf=True
    )),
    ("clf", SGDClassifier(
        loss="hinge",          # linear SVM style
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        class_weight="balanced",
        random_state=42
    ))
])

sgd_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + SGDClassifier",
    sgd_model,
    X_test,
    y_test
)

Word TF-IDF + SGDClassifier
Accuracy: 0.842930654251409
F1 macro: 0.8429302557901891
F1 weighted: 0.842930225139326

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.84      0.84      8163
           1       0.84      0.84      0.84      8161

    accuracy                           0.84     16324
   macro avg       0.84      0.84      0.84     16324
weighted avg       0.84      0.84      0.84     16324

Confusion matrix:
[[6867 1296]
 [1268 6893]]


In [8]:
#Word TF-IDF + Char TF-IDF
word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=100000,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=150000,
    sublinear_tf=True
)

combined_features = FeatureUnion([
    ("word_tfidf", word_tfidf),
    ("char_tfidf", char_tfidf)
])

word_char_ridge_model = Pipeline([
    ("features", combined_features),
    ("clf", RidgeClassifier(
        alpha=1.0,
        class_weight="balanced"
    ))
])

word_char_ridge_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + Char TF-IDF + RidgeClassifier",
    word_char_ridge_model,
    X_test,
    y_test
)

Word TF-IDF + Char TF-IDF + RidgeClassifier
Accuracy: 0.8472188189169321
F1 macro: 0.847218231808502
F1 weighted: 0.8472182685027789

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      8163
           1       0.85      0.85      0.85      8161

    accuracy                           0.85     16324
   macro avg       0.85      0.85      0.85     16324
weighted avg       0.85      0.85      0.85     16324

Confusion matrix:
[[6931 1232]
 [1262 6899]]


In [9]:
word_char_sgd_model = Pipeline([
    ("features", combined_features),
    ("clf", SGDClassifier(
        loss="hinge",
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        class_weight="balanced",
        random_state=42
    ))
])

word_char_sgd_model.fit(X_train, y_train)

evaluate_model(
    "Word TF-IDF + Char TF-IDF + SGDClassifier",
    word_char_sgd_model,
    X_test,
    y_test
)

Word TF-IDF + Char TF-IDF + SGDClassifier
Accuracy: 0.8459323695172751
F1 macro: 0.8459271513241204
F1 weighted: 0.8459270414674224

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      8163
           1       0.84      0.85      0.85      8161

    accuracy                           0.85     16324
   macro avg       0.85      0.85      0.85     16324
weighted avg       0.85      0.85      0.85     16324

Confusion matrix:
[[6857 1306]
 [1209 6952]]


In [10]:
!pip install fasttext -q

In [11]:
import fasttext

In [12]:
fasttext_train_path = "/kaggle/working/fasttext_train.txt"
fasttext_test_path = "/kaggle/working/fasttext_test.txt"

def clean_fasttext_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_fasttext = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_fasttext = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

with open(fasttext_train_path, "w", encoding="utf-8") as f:
    for _, row in train_fasttext.iterrows():
        label = int(row["label"])
        text = clean_fasttext_text(row["text"])
        f.write(f"__label__{label} {text}\n")

with open(fasttext_test_path, "w", encoding="utf-8") as f:
    for _, row in test_fasttext.iterrows():
        label = int(row["label"])
        text = clean_fasttext_text(row["text"])
        f.write(f"__label__{label} {text}\n")

In [13]:
!pip install fasttext -q --force-reinstall

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.5 which is incompatible.


In [14]:
# Train fastText

fasttext_model = fasttext.train_supervised(
    input=fasttext_train_path,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    minn=3,
    maxn=5,
    dim=100,
    loss="softmax",
    verbose=2
)

# Evaluate fastText

def predict_fasttext(model, texts):
    preds = []
    for text in texts:
        text = clean_fasttext_text(text)
        pred = model.predict(text)[0][0]
        pred = int(pred.replace("__label__", ""))
        preds.append(pred)
    return np.array(preds)

fasttext_preds = predict_fasttext(fasttext_model, X_test)

acc = accuracy_score(y_test, fasttext_preds)
f1_macro = f1_score(y_test, fasttext_preds, average="macro")
f1_weighted = f1_score(y_test, fasttext_preds, average="weighted")

results.append({
    "model": "fastText supervised",
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
})

print("=" * 70)
print("fastText supervised")
print("=" * 70)
print("Accuracy:", acc)
print("F1 macro:", f1_macro)
print("F1 weighted:", f1_weighted)
print("\nClassification report:")
print(classification_report(y_test, fasttext_preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, fasttext_preds))

Read 1M words
Number of words:  209350
Number of labels: 2
Progress: 100.0% words/sec/thread:  150292 lr:  0.000000 avg.loss:  0.201383 ETA:   0h 0m 0s  3.7% words/sec/thread:  117205 lr:  0.481256 avg.loss:  0.455952 ETA:   0h 1m43s% words/sec/thread:  130737 lr:  0.466580 avg.loss:  0.428096 ETA:   0h 1m29s 12.9% words/sec/thread:  141478 lr:  0.435596 avg.loss:  0.401166 ETA:   0h 1m17s 24.8% words/sec/thread:  147083 lr:  0.376224 avg.loss:  0.348811 ETA:   0h 1m 4s 28.7% words/sec/thread:  148145 lr:  0.356385 avg.loss:  0.323239 ETA:   0h 1m 0s words/sec/thread:  147666 lr:  0.345057 avg.loss:  0.321796 ETA:   0h 0m58s% words/sec/thread:  146949 lr:  0.286632 avg.loss:  0.297851 ETA:   0h 0m49s words/sec/thread:  148127 lr:  0.171531 avg.loss:  0.254731 ETA:   0h 0m29s% words/sec/thread:  148652 lr:  0.143071 avg.loss:  0.246561 ETA:   0h 0m24s avg.loss:  0.245596 ETA:   0h 0m23s 73.6% words/sec/thread:  148876 lr:  0.131847 avg.loss:  0.243453 ETA:   0h 0m22s  0h 0m 5s


fastText supervised
Accuracy: 0.8255942171036511
F1 macro: 0.825585560950254
F1 weighted: 0.8255857114920522

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.83      0.83      8163
           1       0.83      0.82      0.82      8161

    accuracy                           0.83     16324
   macro avg       0.83      0.83      0.83     16324
weighted avg       0.83      0.83      0.83     16324

Confusion matrix:
[[6796 1367]
 [1480 6681]]


In [15]:
results_df = pd.DataFrame(results).sort_values(
    by="f1_macro",
    ascending=False
)

results_df

,model,accuracy,f1_macro,f1_weighted
2,Word TF-IDF + Char TF-IDF + RidgeClassifier,0.847219,0.847218,0.847218
3,Word TF-IDF + Char TF-IDF + SGDClassifier,0.845932,0.845927,0.845927
0,Word TF-IDF + RidgeClassifier,0.843114,0.843111,0.843111
1,Word TF-IDF + SGDClassifier,0.842931,0.842930,0.842930
4,fastText supervised,0.825594,0.825586,0.825586


In [16]:
import joblib

joblib.dump(ridge_model, "/kaggle/working/ridge_tfidf_model.pkl")
joblib.dump(sgd_model, "/kaggle/working/sgd_tfidf_model.pkl")
joblib.dump(word_char_ridge_model, "/kaggle/working/word_char_ridge_model.pkl")
joblib.dump(word_char_sgd_model, "/kaggle/working/word_char_sgd_model.pkl")

fasttext_model.save_model("/kaggle/working/fasttext_model.bin")

results_df.to_csv("/kaggle/working/model_results.csv", index=False)